# DSID - Big Data : Démonstration technique de création d'un corpus textuel avec Dask 

---

<b><u>Groupe</u> <em>(M1 MIAGE ID Grp. B)</em> : </b>
- PANTANELLA Lucas
- AZRI Asma
- EL QODSI Souleimane

Dans le cadre de la préparation de données pour l'entraînement de Grands Modèles de Langage (LLM), la capacité à traiter des corpus textuels massifs est devenue une nécessité critique. Ce projet exploite le dataset **CORD-19** (extrait de la plateforme Kaggle), une collection de plusieurs milliers d'articles scientifiques sur le COVID-19 (environ **40 Go** de données brutes JSON), pour simuler un pipeline de prétraitement NLP à l'échelle industrielle.

<b><u>Problématique</u> : </b>

Les outils classiques d'analyse de données en Python, tels que **Pandas**, fonctionnent _in-memory_. Sur une machine standard (comme un MacBook Air M1 avec 8 Go de RAM), charger l'intégralité de ce dataset provoquerait inévitablement une saturation de la mémoire (OOM Kill). De plus, le traitement séquentiel sur un seul cœur de processeur serait excessivement long.

<b><u>Solution technique</u> : </b>

Nous utilisons ici Dask, une bibliothèque de calcul parallèle flexible, pour :
- Paralléliser les tâches de nettoyage et d'extraction sur tous les cœurs du CPU (M1).
- Gérer la mémoire en streaming (traitement par partitions) pour ne jamais saturer la RAM.
- Produire un corpus nettoyé et structuré au format Parquet, optimisé pour l'apprentissage machine.

## Code

In [1]:
import time
import json
import glob
import pandas as pd
import os

#### Fonction de preprocessing du dataset utilisée par le cluster Dask

In [2]:
def extract_preprocessing(record):
    # 1. Extraction
    paper_id = record.get("paper_id", "")
    title = record.get("metadata", {}).get("title", "")
    
    abstract_list = record.get("abstract", [])
    abstract_text = " ".join([item["text"] for item in abstract_list if "text" in item])

    body_list = record.get("body_text", [])
    body_text = " ".join([item["text"] for item in body_list if "text" in item])
    
    # 2. Nettoyage immédiat (pour ne pas stocker le texte brut en mémoire)
    # si pas de texte, on retourne None (sera filtré)
    if not isinstance(body_text, str) or len(body_text) < 500:
        return None

    # Regex compilation (rapide)
    text = body_text.lower()
    text = re.sub(r'\S*@\S*\s?', '', text) # Emails
    text = re.sub(r'http\S+', '', text)    # URLs
    text = re.sub(f'[{re.escape(string.punctuation)}]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    return {
        "paper_id": paper_id,
        "title": title,
        "clean_text": text
    }

## Stratégie d'architecture
---

Pour traiter l'intégralité du dataset (~40go) sans saturer la RAM de la machine, nous avons opté pour une architecture de production : 
1. Utilisation d'un script dédié `demo_dask.py` optimisé pour le streaming
2. Cluster `LocalCluster` limitant la mémoire par worker
3. Écriture en streaming au format **Parquet** (plus performant que CSV/JSON)

### Monitoring de l'exécution
Le traitement a été lancé via un terminal de commandes. Voici le Dashboard Dask pendant l'exécution :
- **CPU Utilization (Bleu)** : On voit que les 4 cœurs sont presque utilisées à 100%.
- **Task Stream (Barres centrales)** : Densité maximale, pas de temps mort.
- **Bytes Stored** : La mémoire reste stable (~600 Mo) malgré le volume de données traité.

![Dashboard Dask](./dashboard_screenshot_copy.png)

### Résultat

# Analyse des résultats

---

### Analyse du benchmark Dask

In [3]:
OUTPUT_FOLDER = "./resultats_nettoyage.parquet"

# taille dossier destination
size_bytes = os.path.getsize(OUTPUT_FOLDER) 
# pour un dossier parquet, il faut parfois sommer les fichiers à l'intérieur,

print(f"Dossier de résultats disponible : {OUTPUT_FOLDER} de taille {size_bytes} bytes.")
# !du -sh {OUTPUT_FOLDER} (à décommenter et exécuter si notebook ouvert sur environnement UNIX)

# CHARGEMENT DU RÉSULTAT FINAL
# Pandas peut lire format parquet très rapidement, on peut se permettre de tout charger (tps d'exécution sur Mac M1 : 2.27m)
df_final = pd.read_parquet(OUTPUT_FOLDER, engine='pyarrow')

print(f"Nombre total de documents propres et valides : {len(df_final)}")
print("Aperçu des données :")
pd.set_option('display.max_colwidth', 100)
display(df_final.head()) # 5 premières lignes

Dossier de résultats disponible : ./resultats_nettoyage.parquet de taille 64288 bytes.
Nombre total de documents propres et valides : 399084
Aperçu des données :


,paper_id,title,clean_text
__null_dask_index__,,,
0,261a1408be03a4ba06ce2fef3c6775fecb60e167,Students' Acceptance of Technology-Mediated Teaching -How It Was Influenced During the COVID-19 ...,digital technologies have provided support in diverse policy business and societal application a...
1,efe13333c69a364cb5d4463ba93815e6fc2d91c6,Clinical and epidemiological characteristics of pediatric SARS-CoV-2 infections in China: A mult...,a1111111111 a1111111111 a1111111111 a1111111111 a1111111111 available data 78 in addition signif...
2,9fda06fbd81a070cdecf38a1d7a1248b300187d8,Journal of Clinical VirologyCoV-2 B.1.1.529 (Omicron) Variant of Concern,a novel sars cov 2 variant pango lineage b 1 1 529 was first reported to the world health organi...
3,4fcb95cc0c4ea6d1fa4137a4a087715ed6b68cea,End-tidal carbon dioxide levels during resuscitation and carbon dioxide levels in the immediate ...,improvements in neonatal intensive care have resulted in decreased mortality rates of preterm in...
4,33ed0464cb31e4621ea878050f0a3f2f3d59bfa2,Molecular Research on Platelet Activity in Health and Disease 3.0,zhao and devine 1 exhaustively provide an update on in vitro storage characteristics and in vivo...


#### Commentaire 

L'analyse du dataframe final et des métadonnées de sortie révèle trois succès majeurs de notre architecture distribuée :

1. **Compression et Optimisation du Stockage** : Nous sommes passés d'un dossier source de fichiers JSON pesant plusieurs dizaines de Go (structure verbeuse) à un fichier **Parquet de ~64 Mo** (structure colonnaire compressée). Cette réduction drastique (-90% d'espace disque) facilite le stockage et le chargement futur pour l'entraînement, tout en conservant l'information utile.

2. **Qualité du Nettoyage (NLP)** : L'aperçu des données montre l'efficacité des fonctions de prétraitement appliquées via Dask :

- **Standardisation** : Le texte est uniformisé en minuscules.

- **Réduction du bruit** : Les adresses ema1ils, URL et la ponctuation excessive ont été retirées (visible sur les premières lignes de l'aperçu).

  → Le texte est désormais "propre" et prêt pour une étape de tokenisation ou d'embedding.

3. **Intégrité des Données** : Sur **401 214 fichiers** détectés en entrée, nous retrouvons **399 084 documents valides** en sortie. La différence (~0.5%) correspond aux documents filtrés car vides ou trop courts (< 500 caractères). Cela confirme que le pipeline a correctement filtré les données non pertinentes sans perte accidentelle d'information majeure.

### Benchmark Pandas

In [ ]:
# 1. TEMPS RÉEL DASK (Issu de l'exécution du script démo dask plus haut)
dask_real_time = 422.76 

# 2. ESTIMATION PANDAS (Sur échantillon)
files = glob.glob("./document_parses/pdf_json/*.json")
echantillon_size = 1000
sample_files = files[:echantillon_size] # Echantillon, sinon trop lourd/lent

print(f"Benchmark Pandas sur {echantillon_size} fichiers...")
start = time.time()
data = []
for f_path in sample_files:
    with open(f_path, 'r') as f:
        doc = json.load(f)
        # logique d'extraction simplifiée pour le test, un append est suffisamment similaire en terme de complexité
        data.append(doc) 
    
pandas_duration = time.time() - start

# 3. EXTRAPOLATION
total_files = 401214 # nombre de fichiers total (issu du résultat d'exécution du script python de démo dask à voir plus haut)
projected_pandas = (pandas_duration / 1000) * total_files
projected_hours = projected_pandas / 3600

# 4. CONCLUSION
print("\n--- RÉSULTATS DE LA COMPARAISON ---")
print(f"Temps DASK (Réel)     : {dask_real_time:.2f} s  (~{dask_real_time/60:.1f} min)")
print(f"Temps PANDAS (Estimé) : {projected_pandas:.2f} s (~{projected_hours:.1f} heures)")
print(f"Accélération      : x{projected_pandas / dask_real_time:.1f}")

Benchmark Pandas sur 1000 fichiers...

--- RÉSULTATS DE LA COMPARAISON ---
Temps DASK (Réel)     : 422.76 s  (~7.0 min)
Temps PANDAS (Estimé) : 2302.07 s (~0.6 heures)
Accélération      : x5.4


: 

#### Commentaire et conclusion du benchmark

Les résultats du benchmark illustrent parfaitement l'apport du calcul distribué sur une machine locale performante (Apple M1) :

- **Gain de performance (x5.4)** : Dask a traité l'intégralité du corpus en **~7 minutes**, là où une projection linéaire de Pandas estime le temps à **~38 minutes**. Ce gain s'explique par l'utilisation simultanée des cœurs du processeur (visible sur le Dashboard Dask), contrairement à Pandas qui reste mono-cœur par défaut.

- **La barrière de la RAM (Le facteur critique)** : Il est crucial de noter que cette comparaison est **théorique pour Pandas**. Dans la réalité, charger 40 Go de JSON pour créer un DataFrame Pandas aurait saturé la mémoire vive (RAM) bien avant la fin du traitement, provoquant un crash du noyau.

**Dask** n'est donc pas seulement "plus rapide" ici ; il est l'outil qui rend le traitement possible en gérant les données partition par partition, sans jamais dépasser la capacité physique de la machine.

<b><u>Conclusion</u> : </b> Cette démonstration valide que Dask est une solution intermédiaire idéale entre Pandas (trop limité) et Spark (trop lourd à déployer) pour préparer des corpus NLP sur des infrastructures légères.